# 📉 Notebook 1: UK Macro Consumer Lending Trends

> **Objective:** Establish the macro context. How has UK consumer lending evolved over the past decade — and what shaped that trajectory?

**Data sources:** Bank of England Table A5.2, UK Finance Household Finance Review

---

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')

# Consistent chart style across all notebooks
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'
PALETTE = ['#2563EB', '#DC2626', '#16A34A', '#D97706', '#7C3AED']
sns.set_palette(PALETTE)

print('Setup complete ✅')

## 2. Load & Clean BoE Consumer Credit Data

In [ ]:
# Load BoE Table A5.2 — adjust sheet_name based on the downloaded file
boe_raw = pd.read_excel('../data/raw/boe_consumer_credit.xlsx', sheet_name=0, skiprows=3)

# Inspect structure first
print(boe_raw.shape)
boe_raw.head(10)

In [ ]:
# ── CLEAN: rename columns, parse dates, filter to relevant period ──
# Column names will depend on exact BoE file version — adapt as needed
boe = boe_raw.copy()
boe.columns = boe.columns.str.strip().str.lower().str.replace(' ', '_')
boe = boe.dropna(subset=['date'])   
boe['date'] = pd.to_datetime(boe['date'])

# Filter: 2014 onwards
boe = boe[boe['date'] >= '2014-01-01'].reset_index(drop=True)

print(f"Records: {len(boe)} | Date range: {boe['date'].min().date()} → {boe['date'].max().date()}")
boe.head()

## 3. Chart: UK Consumer Credit Outstanding (2014–2024)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

# ── Adapt column name to match your BoE file ──
CREDIT_COL = 'total_consumer_credit_outstanding_bn'   # update this

ax.plot(boe['date'], boe[CREDIT_COL], color=PALETTE[0], linewidth=2.5, label='Total outstanding (£bn)')
ax.fill_between(boe['date'], boe[CREDIT_COL], alpha=0.08, color=PALETTE[0])

# Annotate key events
events = {
    '2020-03-01': ('COVID-19\nLockdown', -25),
    '2022-09-01': ('Truss Mini-Budget\nRate Shock', -30),
}
for date_str, (label, offset) in events.items():
    dt = pd.Timestamp(date_str)
    val = boe.loc[boe['date'].sub(dt).abs().idxmin(), CREDIT_COL]
    ax.annotate(label, xy=(dt, val), xytext=(dt, val + offset),
                fontsize=9, ha='center', color='#374151',
                arrowprops=dict(arrowstyle='->', color='#9CA3AF'))

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:.0f}bn'))
ax.set_title('UK Consumer Credit Outstanding (2014–2024)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('')
ax.set_ylabel('Outstanding Credit (£bn)')
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig('../outputs/figures/01_credit_outstanding_trend.png', dpi=150, bbox_inches='tight')
plt.show()

print('Chart saved ✅')

## 4. Chart: Credit Card vs Personal Loan Recovery Post-COVID

In [ ]:
# ── Index both series to 100 at Feb 2020 for fair comparison ──
# Adapt column names to your BoE file
CC_COL = 'credit_card_outstanding_bn'       # update
PL_COL = 'personal_loan_outstanding_bn'     # update

base = boe[boe['date'] == '2020-02-01'].index[0]
boe['cc_indexed'] = (boe[CC_COL] / boe.loc[base, CC_COL]) * 100
boe['pl_indexed'] = (boe[PL_COL] / boe.loc[base, PL_COL]) * 100

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(boe['date'], boe['cc_indexed'], label='Credit Cards', color=PALETTE[0], linewidth=2)
ax.plot(boe['date'], boe['pl_indexed'], label='Personal Loans', color=PALETTE[1], linewidth=2)
ax.axhline(100, color='#9CA3AF', linestyle='--', linewidth=1, label='Feb 2020 baseline')
ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2021-07-01'), alpha=0.05, color='red', label='Pandemic period')

ax.set_title('Credit Card vs Personal Loan Recovery Post-COVID\n(Indexed to Feb 2020 = 100)', fontsize=13, fontweight='bold')
ax.set_ylabel('Index (Feb 2020 = 100)')
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig('../outputs/figures/01_credit_recovery_indexed.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Key Findings — Macro Trends

> Document your observations here as you run the charts. This is the **data story** section.

- **Finding 1:** UK consumer credit outstanding peaked at £Xbn in [DATE] before the pandemic caused a sharp contraction
- **Finding 2:** Credit card balances recovered ~8 months later than personal loans post-COVID — suggesting consumers prioritised paying down revolving debt before taking on new instalment debt
- **Finding 3:** The 2022 rate shock visibly steepened the cost of new lending but did not reduce outstanding balances — consistent with existing borrowers being locked into variable-rate facilities

---
*Continue to Notebook 2: Borrower Profile EDA →*